# Lecture 4 — Building a Flexible NLA Intrinsic-Alignment Model

In the previous notebook we used **PyCCL** to create a cosmology and calculate the
nonlinear matter power spectrum $P_{\rm m}(k,z)$. In this notebook we will turn
that matter spectrum into a simple family of **three-dimensional intrinsic-alignment
(IA) power spectra**.

## Learning goals

By the end, you should be able to:

1. explain why a tidal gravitational field can influence galaxy orientations;
2. understand the two NLA spectra $P_{\delta I}$ and $P_{II}$;
3. describe the roles of amplitude, redshift, luminosity, and scale parameters;
4. understand how a luminosity-function average motivates a simpler effective
   redshift law;
5. generate many IA spectra and store them as a machine-learning dataset.

The scale-dependent extension used here is deliberately **phenomenological**:
it is a controlled teaching model, not the TATT model.

## 1. The big picture

A galaxy forms inside the cosmic web. Gravity does not pull equally strongly in every
direction: nearby matter can stretch the local gravitational environment more along one
direction than another. This directional stretching is called the **tidal field**.

A simple IA hypothesis is:

> The coherent part of a galaxy's intrinsic shape responds to the surrounding tidal field.

We summarize that response with an IA amplitude $F$:

$$
\text{intrinsic shape} \approx F \times \text{matter tidal field}.
$$

If the matter field has power spectrum $P_{\rm m}$, this leads to

$$
P_{\delta I}(k,z)=F(k,z)\,P_{\rm m}(k,z),
$$

$$
P_{II}(k,z)=F^2(k,z)\,P_{\rm m}(k,z).
$$

- $P_{\delta I}$: correlation between matter density and intrinsic galaxy shape;
- $P_{II}$: correlation between two intrinsic-shape fields.

Under the sign convention used here, positive $A_0$ gives negative
$P_{\delta I}$, while $P_{II}$ remains non-negative.

## 2. Imports

This notebook uses the same PyCCL environment as the previous lecture.

In [ ]:
import json
from pathlib import Path

import numpy
import pyccl
from matplotlib import pyplot

In [2]:
path = Path.cwd().parent
code_path = path / 'Code'
data_path = path / 'Data'
figure_path = path / 'Figure'
print(code_path, data_path, figure_path)

/Users/s2227120/Downloads/IAFlowCloud/Code /Users/s2227120/Downloads/IAFlowCloud/Data /Users/s2227120/Downloads/IAFlowCloud/Figure


## 3. A fiducial cosmology

We initially keep cosmology fixed so that we can understand the IA parameters one at a
time. The Eisenstein–Hu transfer function avoids requiring a separate CAMB installation.
PyCCL uses distances in Mpc and wavenumbers in $\mathrm{Mpc}^{-1}$.

In [3]:
with open(data_path / 'Planck.json', 'r') as file:
    parameter = json.load(file)
print(parameter)

{'H': 0.6731999996749277, 'W0': -1.0, 'WA': 0.0, 'NS': 0.966, 'AS': 2.101e-09, 'MNU': 0.06, 'TCMB': 2.7255, 'NEFF': 3.0440016274835076, 'SIGMA8': 0.812, 'OMEGAM': 0.314419, 'OMEGAB': 0.04939, 'OMEGAC': 0.265029, 'OMEGAD': 0.68408, 'OMEGAN': 0.0014214549162596109, 'OMEGAR': 7.973409482179201e-05, 'OMEGAK': -1.8901108145286116e-07}


In [ ]:
cosmology = pyccl.cosmology.Cosmology(
    h = parameter['H'],
    w0 = parameter['W0'],
    wa = parameter['WA'],
    A_s = parameter['AS'], 
    n_s = parameter['NS'], 
    m_nu = parameter['MNU'], 
    T_CMB = parameter['TCMB'],
    Omega_k = parameter['OMEGAK'], 
    Omega_c = parameter['OMEGAC'], 
    Omega_b = parameter['OMEGAB'], 
    mass_split='normal', transfer_function = 'boltzmann_camb', 
    extra_parameters = {'camb': {'kmax': 100, 'lmax': 5000, 'halofit_version': 'mead2020_feedback', 'HMCode_logT_AGN': 7.8}}
)

Omega_m = 0.3158456489377449


## 4. From linear alignment to nonlinear alignment

In the **linear-alignment (LA)** model, the IA response is multiplied by the linear
matter power spectrum:

$$
P_{\delta I}^{\rm LA}=F\,P_{\rm lin},
\qquad
P_{II}^{\rm LA}=F^2\,P_{\rm lin}.
$$

The **nonlinear-alignment (NLA)** prescription keeps the same simple response but
replaces the linear matter spectrum with a nonlinear one:

$$
P_{\delta I}^{\rm NLA}=F\,P_{\rm nl},
\qquad
P_{II}^{\rm NLA}=F^2\,P_{\rm nl}.
$$

NLA is a useful empirical baseline. It is not a complete theory of nonlinear galaxy
formation. More advanced models, such as TATT, introduce additional tidal-field terms.

## 5. Constructing the response one factor at a time

We use

$$
F(k,z)=
-A_0 C_0
\frac{\Omega_m}{D(z)}
R_z(z;\eta)
R_L(z;\lambda_L,\kappa_L)
S(k;q,k_t,n).
$$

The factors have separate meanings:

| Factor | Meaning |
|---|---|
| $A_0$ | overall IA strength and sign at the pivots |
| $C_0$ | conventional normalization, approximately 0.0134 |
| $\Omega_m/D(z)$ | standard cosmology and growth scaling |
| $R_z$ | extra physical redshift evolution |
| $R_L$ | effective luminosity-population evolution |
| $S(k)$ | phenomenological transition from large to small scales |

We normalize $R_z(z_p)=R_L(z_p)=1$. We also have $S(k\ll k_t)=1$.
Therefore $A_0$ remains easy to interpret.

## 6. Extra redshift evolution

The conventional growth factor already introduces redshift evolution. We allow an
additional empirical power law:

$$
R_z(z;\eta)
=
\left(\frac{1+z}{1+z_p}\right)^\eta.
$$

- $\eta=0$: no additional evolution;
- $\eta>0$: stronger response at high redshift;
- $\eta<0$: weaker response at high redshift.

The pivot $z_p$ prevents a change in $\eta$ from changing the amplitude exactly at
$z=z_p$.

In [ ]:
C0 = 0.0134
Z_PIVOT = 0.5

def redshift_factor(z, eta, z_pivot=Z_PIVOT):
    z = numpy.asarray(z)
    return ((1.0 + z) / (1.0 + z_pivot))**eta

SyntaxError: invalid decimal literal (832127471.py, line 1)

## 7. A simple effective luminosity factor

Physically, the luminosity contribution is an average over the galaxies in a redshift
bin:

$$
R_L(z)\ \propto\
\left\langle\left(\frac{L}{L_p}\right)^\beta\right\rangle_z .
$$

A Schechter luminosity function gives this average analytically, but most of its
redshift trend is usually driven by the evolving characteristic luminosity
$L_\star(z)$. If locally

$$
\frac{L_\star(z)}{L_\star(z_p)}
\approx
\left(\frac{1+z}{1+z_p}\right)^\gamma,
$$

then the leading luminosity factor is approximately a power law with slope
$\lambda_L\simeq\beta\gamma$.

Instead of modelling the full luminosity function, we use the flexible redshift-only
approximation

$$
\boxed{
R_L(z;\lambda_L,\kappa_L)
=
\exp\left[\lambda_L x+\kappa_Lx^2\right],
\qquad
x=\ln\left(\frac{1+z}{1+z_p}\right).
}
$$

Equivalently,

$$
R_L=
\left(\frac{1+z}{1+z_p}\right)^{
\lambda_L+\kappa_L\ln[(1+z)/(1+z_p)]
}.
$$

- $\lambda_L$: leading power-law slope, mainly representing the evolution of
  $L_\star$ and the luminosity exponent $\beta$;
- $\kappa_L$: gentle curvature caused by changes in sample selection or by the
  slowly varying remainder of the luminosity-function average;
- $\lambda_L=\kappa_L=0$: no luminosity-population evolution.

The exponential keeps $R_L$ positive, and the pivot automatically gives
$R_L(z_p)=1$.

**Important degeneracy:** $R_z\propto[(1+z)/(1+z_p)]^\eta$ and the
$\lambda_L$ part of $R_L$ have the same mathematical shape. Power spectra alone
therefore constrain $\eta+\lambda_L$, not the two slopes separately, unless external
luminosity information is supplied. Keeping both is useful for teaching and forward
modelling, but they should not be interpreted as independently measured parameters.

In [ ]:
def luminosity_factor(
    z,
    lambda_L=0.0,
    kappa_L=0.0,
    z_pivot=Z_PIVOT,
):
    """Effective luminosity-population factor, normalized at z_pivot."""
    z = numpy.asarray(z)
    x = numpy.log((1.0 + z) / (1.0 + z_pivot))
    return numpy.exp(lambda_L * x + kappa_L * x**2)

### 7.1 Visualize the effective luminosity factor

The left panel changes the main power-law slope. The right panel changes only the
curvature. Every curve crosses one at the pivot redshift.

In [ ]:
z_plot = numpy.linspace(0.0, 2.0, 200)

fig, axes = pyplot.subplots(1, 2, figsize=(13, 5))

for lambda_L in [-2.0, -1.0, 0.0, 1.0, 2.0]:
    axes[0].plot(
        z_plot,
        luminosity_factor(z_plot, lambda_L=lambda_L, kappa_L=0.0),
        label=rf"$\lambda_L={lambda_L:.1f}$",
    )

for kappa_L in [-2.0, -1.0, 0.0, 1.0, 2.0]:
    axes[1].plot(
        z_plot,
        luminosity_factor(z_plot, lambda_L=0.0, kappa_L=kappa_L),
        label=rf"$\kappa_L={kappa_L:.1f}$",
    )

for ax in axes:
    ax.axvline(Z_PIVOT, color="black", linestyle=":", label="pivot redshift")
    ax.axhline(1.0, color="grey", linewidth=1)
    ax.set_xlabel("redshift z")
    ax.set_ylabel(r"$R_L(z)$")
    ax.legend()

axes[0].set_title("Leading power-law slope")
axes[1].set_title("Curvature around the pivot")
fig.tight_layout()
pyplot.show()

## 8. A smooth transition between large and small scales

Standard NLA has no extra scale-dependent bias. We add a smooth phenomenological
transition:

$$
\boxed{
S(k;q,k_t,n)
=
1+
q\frac{(k/k_t)^n}{1+(k/k_t)^n}.
}
$$

Its limits are

$$
S(k\ll k_t)\rightarrow1,
\qquad
S(k\gg k_t)\rightarrow1+q.
$$

At $k=k_t$,

$$
S(k_t)=1+\frac{q}{2}.
$$

Parameter meanings:

- $q$: strength of the small-scale enhancement or suppression;
- $k_t$: transition wavenumber;
- $n$: sharpness of the transition.

Setting $q=0$ recovers standard NLA. Requiring $q>-1$ keeps $S(k)$ positive.

In [ ]:
def scale_transition(k, q, k_transition, sharpness):
    """Smooth transition S(k) from 1 to 1+q."""
    if q <= -1:
        raise ValueError("Use q > -1 so the high-k response stays positive.")
    if k_transition <= 0 or sharpness <= 0:
        raise ValueError("k_transition and sharpness must be positive.")

    ratio_power = (numpy.asarray(k) / k_transition)**sharpness
    return 1.0 + q * ratio_power / (1.0 + ratio_power)


k_plot = numpy.logspace(-3, 0.5, 300)

fig, axes = pyplot.subplots(1, 3, figsize=(15, 4.5))

for q in [-0.5, 0.0, 0.5, 1.5]:
    axes[0].plot(
        k_plot,
        scale_transition(k_plot, q, 0.2, 2.0),
        label=rf"$q={q}$",
    )

for k_t in [0.05, 0.2, 0.8]:
    axes[1].plot(
        k_plot,
        scale_transition(k_plot, 1.0, k_t, 2.0),
        label=rf"$k_t={k_t}$",
    )

for n in [1.0, 2.0, 4.0]:
    axes[2].plot(
        k_plot,
        scale_transition(k_plot, 1.0, 0.2, n),
        label=rf"$n={n}$",
    )

for ax in axes:
    ax.set_xscale("log")
    ax.set_xlabel(r"$k$ [Mpc$^{-1}$]")
    ax.set_ylabel(r"$S(k)$")
    ax.axhline(1.0, color="grey", linewidth=1)
    ax.legend()

axes[0].set_title("Small-scale strength")
axes[1].set_title("Transition location")
axes[2].set_title("Transition sharpness")
fig.tight_layout()
pyplot.show()

## 9. Assemble the complete IA response

We now combine:

$$
\frac{1}{D(z)}
\times
R_z(z;\eta)
\times
R_L(z;\lambda_L,\kappa_L)
\times
S(k;q,k_t,n).
$$

This function returns both the complete two-dimensional response $F(z,k)$ and its
separate factors. Keeping the factors available makes the model easier to debug and
explain.

In [ ]:
def ia_response(
    cosmo,
    z,
    k,
    A0=1.0,
    eta=0.0,
    lambda_L=0.0,
    kappa_L=0.0,
    q=0.0,
    k_transition=0.2,
    sharpness=2.0,
):
    z = numpy.asarray(z)
    k = numpy.asarray(k)

    a = 1.0 / (1.0 + z)
    growth = pyccl.growth_factor(cosmo, a)
    R_z = redshift_factor(z, eta)
    R_L = luminosity_factor(z, lambda_L, kappa_L)
    S_k = scale_transition(k, q, k_transition, sharpness)

    redshift_amplitude = (
        -A0
        * C0
        * cosmo["Omega_m"]
        / growth
        * R_z
        * R_L
    )

    F = redshift_amplitude[:, None] * S_k[None, :]
    factors = {
        "growth": growth,
        "R_z": R_z,
        "R_L": R_L,
        "S_k": S_k,
        "redshift_amplitude": redshift_amplitude,
    }
    return F, factors

## 10. Generate fiducial 3D IA spectra

We evaluate the nonlinear matter spectrum on the same $(z,k)$ grid and then apply the
IA response. The resulting arrays contain one curve per redshift.

In [ ]:
k = numpy.logspace(-3, 0.5, 160)
z = numpy.asarray([0.2, 0.5, 0.8, 1.2, 1.6, 2.0])

P_m = numpy.vstack([
    pyccl.nonlin_matter_power(
        cosmology,
        k,
        1.0 / (1.0 + z_value),
    )
    for z_value in z
])

fiducial_parameters = {
    "A0": 1.0,
    "eta": 0.0,
    "lambda_L": 1.0,
    "kappa_L": 0.0,
    "q": 1.0,
    "k_transition": 0.2,
    "sharpness": 2.0,
}

F, factors = ia_response(
    cosmology,
    z,
    k,
    **fiducial_parameters,
)

P_deltaI = F * P_m
P_II = F**2 * P_m

print("P_m shape:      ", P_m.shape)
print("F shape:        ", F.shape)
print("P_deltaI shape: ", P_deltaI.shape)
print("P_II shape:     ", P_II.shape)

### 10.1 Inspect the separate redshift factors

$1/D(z)$, $R_z$, and $R_L$ can all change the amplitude with redshift, but they
represent different assumptions. This also means their parameters may be partly
degenerate in a future inference analysis.

In [ ]:
fig, axes = pyplot.subplots(1, 3, figsize=(15, 4.5))

axes[0].plot(z, 1.0 / factors["growth"], marker="o")
axes[0].set_title("Standard growth scaling")
axes[0].set_ylabel(r"$1/D(z)$")

axes[1].plot(z, factors["R_z"], marker="o")
axes[1].set_title("Extra redshift factor")
axes[1].set_ylabel(r"$R_z(z;\eta)$")

axes[2].plot(z, factors["R_L"], marker="o")
axes[2].set_title("Effective luminosity factor")
axes[2].set_ylabel(r"$R_L(z;\lambda_L,\kappa_L)$")

for ax in axes:
    ax.set_xlabel("redshift z")
    ax.axvline(Z_PIVOT, color="grey", linestyle=":")

fig.tight_layout()
pyplot.show()

### 10.2 Plot the spectra

For positive $A_0$, our convention gives $P_{\delta I}<0$. A logarithmic axis
cannot display negative numbers, so the middle panel shows $-P_{\delta I}$ and labels
this explicitly. We never silently discard the sign in the saved data.

In [ ]:
fig, axes = pyplot.subplots(1, 3, figsize=(17, 5))

for index, z_value in enumerate(z):
    label = rf"$z={z_value:.1f}$"
    axes[0].plot(k, P_m[index], label=label)
    axes[1].plot(k, -P_deltaI[index], label=label)
    axes[2].plot(k, P_II[index], label=label)

titles = [
    "Nonlinear matter spectrum",
    "Matter–intrinsic spectrum",
    "Intrinsic–intrinsic spectrum",
]
ylabels = [
    r"$P_{\rm m}(k,z)$ [Mpc$^3$]",
    r"$-P_{\delta I}(k,z)$ [Mpc$^3$]",
    r"$P_{II}(k,z)$ [Mpc$^3$]",
]

for ax, title, ylabel in zip(axes, titles, ylabels):
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel(r"$k$ [Mpc$^{-1}$]")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(fontsize=8)

fig.tight_layout()
pyplot.show()

## 11. What does each parameter do?

The most direct diagnostic is

$$
\left|\frac{P_{\delta I}}{P_{\rm m}}\right|=|F(k,z)|.
$$

This removes the matter-spectrum shape and displays the IA response itself.

| Parameter | Main visible effect |
|---|---|
| $A_0$ | rescales everything; changing its sign changes $P_{\delta I}$'s sign |
| $\eta$ | changes the extra physical redshift trend relative to the pivot |
| $\lambda_L$ | changes the leading luminosity-population redshift slope |
| $\kappa_L$ | curves the luminosity evolution around the pivot |
| $q$ | changes the high-$k$ amplitude |
| $k_t$ | moves the transition left or right |
| $n$ | makes the transition smoother or sharper |

In [ ]:
z_test = numpy.asarray([1.2])

experiments = [
    ("A0", [0.5, 1.0, 2.0]),
    ("eta", [-2.0, 0.0, 2.0]),
    ("lambda_L", [-2.0, 0.0, 2.0]),
    ("kappa_L", [-2.0, 0.0, 2.0]),
    ("q", [-0.5, 0.0, 1.5]),
    ("k_transition", [0.05, 0.2, 0.8]),
    ("sharpness", [1.0, 2.0, 4.0]),
]

baseline = {
    "A0": 1.0,
    "eta": 0.0,
    "lambda_L": 1.0,
    "kappa_L": 0.0,
    "q": 1.0,
    "k_transition": 0.2,
    "sharpness": 2.0,
}

fig, axes = pyplot.subplots(2, 4, figsize=(19, 9))

for ax, (parameter_name, values) in zip(axes.flat, experiments):
    for value in values:
        parameters = baseline.copy()
        parameters[parameter_name] = value
        response, _ = ia_response(
            cosmology,
            z_test,
            k,
            **parameters,
        )
        ax.plot(
            k,
            numpy.abs(response[0]),
            label=f"{parameter_name}={value}",
        )

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel(r"$k$ [Mpc$^{-1}$]")
    ax.set_ylabel(r"$|P_{\delta I}/P_{\rm m}|$")
    ax.set_title(f"Vary {parameter_name}")
    ax.legend(fontsize=8)

axes.flat[-1].axis("off")
fig.suptitle(
    r"One parameter at a time, evaluated at $z=1.2$",
    fontsize=15,
)
fig.tight_layout()
pyplot.show()

## 12. Sample parameters and create an ML-ready dataset

Each sampled parameter set creates two surfaces:

$$
P_{\delta I}(z,k)
\quad\text{and}\quad
P_{II}(z,k).
$$

We store them as two channels:

$$
X.\mathrm{shape}
=
(N_{\rm models},\,2,\,N_z,\,N_k).
$$

This example uses independent uniform sampling for clarity. A larger project should use
Latin-hypercube or Sobol sampling for more uniform coverage.

The ranges below are broad teaching ranges, not observational priors.

In [ ]:
rng = numpy.random.default_rng(2026)
number_of_models = 300

parameter_names = [
    "A0",
    "eta",
    "lambda_L",
    "kappa_L",
    "q",
    "k_transition",
    "sharpness",
]

theta = numpy.column_stack([
    rng.uniform(0.2, 3.0, number_of_models),          # A0
    rng.uniform(-2.0, 2.0, number_of_models),         # eta
    rng.uniform(-2.0, 2.0, number_of_models),         # lambda_L
    rng.uniform(-2.0, 2.0, number_of_models),         # kappa_L
    rng.uniform(-0.5, 2.0, number_of_models),         # q
    10.0**rng.uniform(-2.0, -0.2, number_of_models),  # k_t
    rng.uniform(1.0, 4.0, number_of_models),          # n
])

k_training = numpy.logspace(-3, 0.5, 96)
z_training = numpy.asarray([0.2, 0.5, 0.8, 1.2, 1.6, 2.0])

P_m_training = numpy.vstack([
    pyccl.nonlin_matter_power(
        cosmology,
        k_training,
        1.0 / (1.0 + z_value),
    )
    for z_value in z_training
])

P_deltaI_training = numpy.empty(
    (number_of_models, len(z_training), len(k_training))
)
P_II_training = numpy.empty_like(P_deltaI_training)

for model_index, values in enumerate(theta):
    parameters = dict(zip(parameter_names, values))
    response, _ = ia_response(
        cosmology,
        z_training,
        k_training,
        **parameters,
    )

    P_deltaI_training[model_index] = response * P_m_training
    P_II_training[model_index] = response**2 * P_m_training

X = numpy.stack(
    [P_deltaI_training, P_II_training],
    axis=1,
)

print("theta shape:", theta.shape)
print("X shape:    ", X.shape)
print("channels:    [P_deltaI, P_II]")

### 12.1 Inspect a few randomly generated models

These plots are a basic quality check. A training dataset should be inspected before it
is given to PCA or a neural network.

In [ ]:
selected_models = [0, 1, 2, 3, 4]
z_index = 2

fig, axes = pyplot.subplots(1, 2, figsize=(13, 5))

for model_index in selected_models:
    axes[0].plot(
        k_training,
        -P_deltaI_training[model_index, z_index],
        label=f"model {model_index}",
    )
    axes[1].plot(
        k_training,
        P_II_training[model_index, z_index],
        label=f"model {model_index}",
    )

axes[0].set_title(
    rf"$-P_{{\delta I}}$ at $z={z_training[z_index]}$"
)
axes[1].set_title(
    rf"$P_{{II}}$ at $z={z_training[z_index]}$"
)

for ax in axes:
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel(r"$k$ [Mpc$^{-1}$]")
    ax.set_ylabel(r"power spectrum [Mpc$^3$]")
    ax.legend(fontsize=8)

fig.tight_layout()
pyplot.show()

### 12.2 Save the raw dataset

Save raw signed spectra before applying logarithms or normalization. Preprocessing belongs
in the next lecture and should be reproducible from the saved physical quantities.

In [ ]:
if Path.cwd().name == "Code":
    output_directory = Path.cwd().parent / "Data"
else:
    output_directory = Path.cwd() / "Data"

output_directory.mkdir(parents=True, exist_ok=True)
output_file = output_directory / "nla_ia_training_spectra.npz"

numpy.savez_compressed(
    output_file,
    k=k_training,
    z=z_training,
    theta=theta,
    parameter_names=numpy.asarray(parameter_names),
    P_m=P_m_training,
    P_deltaI=P_deltaI_training,
    P_II=P_II_training,
    X=X,
)

print("Saved:", output_file.resolve())

## 13. Interpretation and limitations

This model is useful because each source of variation is visible and controllable:

$$
\text{cosmology}
\times
\text{growth}
\times
\text{extra redshift evolution}
\times
\text{effective luminosity evolution}
\times
\text{scale transition}.
$$

But remember:

1. the NLA replacement $P_{\rm lin}\rightarrow P_{\rm nl}$ is empirical;
2. the transition $S(k)$ is our phenomenological extension, not TATT;
3. $R_L$ summarizes a luminosity-function average rather than predicting it from a
   calibrated survey model;
4. $\eta$ and $\lambda_L$ are exactly degenerate in this factorized model, while
   $\kappa_L$ adds separately identifiable curvature;
5. broad phenomenological ranges are useful for training, but scientific priors should
   come from galaxy-population measurements;
6. a later scientific analysis should test whether reconstructed spectra preserve
   downstream weak-lensing observables.

# Assignment

1. Set $q=0$ and confirm that the scale transition disappears.
2. Change one parameter at a time and describe what happens physically.
3. Explain why $\lambda_L=\kappa_L=0$ makes $R_L(z)=1$.
4. Compare a change in $\eta$ with the same change in $\lambda_L$. Why do their
   curves agree?
5. Vary $\kappa_L$ and explain how curvature differs from a simple power law.
6. Generate at least 100 parameter combinations and verify the shapes of `theta` and `X`.
7. Write a short paragraph answering:

   > Why can a changing galaxy population imitate physical redshift evolution of
   intrinsic alignments?

## Final takeaway

A sampled power spectrum is a long numerical vector. By varying physically meaningful
parameters, we have produced a family of related vectors. That family is the input to
the compression models introduced in the next phase of the project.

## Optional further reading

These papers are not required for completing the notebook:

- [Bridle & King (2007)](https://arxiv.org/abs/0705.0166) introduced the commonly
  used nonlinear-alignment prescription for weak-lensing forecasts.
- [Joachimi et al. (2011)](https://www.aanda.org/articles/aa/pdf/2011/03/aa15621-10.pdf)
  studied observational redshift and luminosity dependence of intrinsic alignments.
- [Blazek et al. (2019)](https://arxiv.org/abs/1708.09247) developed the more physical
  tidal-alignment and tidal-torquing (TATT) framework.

Our function $S(k)$ is a transparent phenomenological interpolation. It should be
calibrated or replaced before it is used for scientific parameter inference.